# torch.cuda 
- 是 PyTorch 框架中专门用于 NVIDIA CUDA GPU 加速 的核心子模块。
- <font color='red'>它提供了从底层 CUDA 运行时控制到高层张量运算的完整接口</font>，是深度学习模型在 GPU 上训练和推理的基础。
- 下面从架构设计、核心 API、内存管理、多 GPU 支持、性能分析等多个维度进行详细介绍。

## 架构总览
torch.cuda 的核心职责：
1. <font color='red'>设备管理</font>：检测 GPU、设置当前设备、多 GPU 切换
2. <font color='red'>张量运算</font>：在 GPU 上创建和执行张量操作
3. <font color='red'>内存管理</font>：显存分配、缓存池、内存统计、OOM 诊断
4. <font color='red'>流与事件</font>：异步执行、同步控制、性能计时
5. <font color='red'>混合精度</font>：自动混合精度 (AMP) 训练支持
6. <font color='red'>性能分析</font>：NVTX 标记、Profiler 集成. 

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│                    torch.cuda (Python API)                   │
├─────────────────────────────────────────────────────────────┤
│  torch.cuda.Tensor    torch.cuda.Stream   torch.cuda.Event   │
│  torch.cuda.memory    torch.cuda.nvtx    torch.cuda.amp     │
├─────────────────────────────────────────────────────────────┤
│              libtorch_cuda (C++ / ATen 后端)                │
├─────────────────────────────────────────────────────────────┤
│              CUDA Runtime / cuDNN / cuBLAS                  │
├─────────────────────────────────────────────────────────────┤
│                    NVIDIA GPU Hardware                      │
└─────────────────────────────────────────────────────────────┘

## 核心 API 详解
### 设备管理

In [ ]:
import torch

# 检测 CUDA 可用性
torch.cuda.is_available()           # True/False，是否有 NVIDIA GPU
torch.cuda.device_count()             # 系统中 GPU 数量
torch.cuda.get_device_name(0)         # "NVIDIA A100-SXM4-80GB"
torch.cuda.get_device_capability(0)   # (8, 0) -> Compute Capability 8.0 (Ampere)

# 当前设备设置
torch.cuda.current_device()           # 获取当前活跃 GPU 索引
torch.cuda.set_device(0)              # 设置当前进程使用的 GPU
torch.cuda.device(0)                  # 返回 torch.device('cuda:0') 的上下文管理器

# 设备属性查询
props = torch.cuda.get_device_properties(0)
print(props.name)           # GPU 型号
print(props.total_memory)   # 总显存（字节）
print(props.multi_processor_count)  # SM 数量

##### 多 GPU 场景下的最佳实践：

In [ ]:
# 方式1：显式指定 device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
tensor = torch.randn(3, 3, device=device)

# 方式2：.cuda() 方法（旧式，不推荐用于新代码）
tensor = torch.randn(3, 3).cuda(0)

# 方式3：.to() 方法（推荐，支持 device 无关代码）
tensor = torch.randn(3, 3).to('cuda:0')

### 张量创建与运算
- 关键特性：PyTorch 的运算遵循 "张量在哪，运算就在哪" 的原则。
- GPU 张量的运算由 CUDA kernel 在 GPU 上异步执行。

In [ ]:
# 在 GPU 上直接创建张量
x = torch.randn(1024, 1024, device='cuda')

# 所有 PyTorch 运算自动在对应设备上执行
y = torch.matmul(x, x.T)        # 在 GPU 上执行矩阵乘法
z = torch.nn.functional.relu(y)  # 在 GPU 上执行激活函数

# 跨设备操作会报错
cpu_tensor = torch.randn(3, 3)
# gpu_tensor + cpu_tensor  # RuntimeError: 期望两个张量在同一设备

### CUDA Streams（流）

CUDA Stream 是一系列按顺序执行的 GPU 操作序列。<font color='red'>不同 Stream 之间的操作可以并发执行</font>

典型应用场景：
- 计算与数据传输重叠：<font color='red'>在一个流上计算，另一个流上做 H2D/D2H 拷贝</font>
- 多任务并行：<font color='red'>独立的前向/反向传播任务在不同流上并发- </font>

In [ ]:
# 默认流（每个设备有一个）
default_stream = torch.cuda.default_stream()

# 创建新流
s1 = torch.cuda.Stream()
s2 = torch.cuda.Stream()

# 在指定流上执行操作
with torch.cuda.stream(s1):
    a = torch.matmul(x, w1)

with torch.cuda.stream(s2):
    b = torch.matmul(y, w2)

# 流同步
s1.synchronize()  # 等待 s1 中所有操作完成
torch.cuda.synchronize()  # 等待当前设备所有流完成

# 流之间的事件同步
e1 = torch.cuda.Event()
e1.record(s1)       # 在 s1 上记录事件
e1.wait(s2)         # s2 等待 e1 完成后再执行

### CUDA Events（事件）

In [ ]:
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()          # 记录开始事件
# ... 执行 GPU 操作 ...
end.record()            # 记录结束事件

torch.cuda.synchronize()  # 必须同步后才能获取准确时间
elapsed_ms = start.elapsed_time(end)  # 返回毫秒

## 显存管理
###  显存分配机制
PyTorch 使用 CUDA Memory Allocator 管理显存，采用缓存池策略：

- 这种设计的优点：避免频繁的 cudaMalloc/cudaFree 系统调用开销。
- <font color='red'>缺点：nvidia-smi 看到的显存占用可能远大于实际活跃张量所需。</font>

In [ ]:
┌─────────────────────────────────────────┐
│           PyTorch CUDA Allocator         │
├─────────────────────────────────────────┤
│  1. 向 CUDA Runtime 申请大块显存         │
│  2. 分割成小块满足张量分配需求            │
│  3. 释放时不立即归还，放入缓存池复用      │
│  4. 缓存池满了才归还给 CUDA Runtime      │
└─────────────────────────────────────────┘

### 显存监控 API

In [ ]:
# 当前已分配显存（由 PyTorch 管理）
allocated = torch.cuda.memory_allocated()      # 字节
allocated_mb = allocated / 1024**2

# 缓存池中保留的显存（已分配但未使用）
reserved = torch.cuda.memory_reserved()        # 字节

# 总计可用显存
total = torch.cuda.get_device_properties(0).total_memory

# 峰值统计
max_allocated = torch.cuda.max_memory_allocated()
max_reserved = torch.cuda.max_memory_reserved()

# 内存摘要（PyTorch 2.0+）
print(torch.cuda.memory_summary())

### 显存释放与缓存清空

⚠️ 注意：empty_cache() 会强制释放缓存池中的显存块，但：
- 不会释放仍在引用的张量
- 会导致后续分配重新触发 cudaMalloc，有性能开销
- 通常只在 OOM 后诊断或显存碎片严重时使用

In [ ]:
# 删除张量
del tensor
torch.cuda.empty_cache()  # 强制清空未使用的缓存池（谨慎使用）

# 重置峰值统计
torch.cuda.reset_peak_memory_stats()

# 上下文管理器：临时清空缓存
with torch.cuda.device(0):
    torch.cuda.empty_cache()

### 显存分配器配置

In [ ]:
# 设置显存分配器的配置
torch.cuda.set_per_process_memory_fraction(0.8)  # 限制 PyTorch 最多使用 80% 显存

# 使用 expandable_segments 减少显存碎片（PyTorch 2.1+）
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

## 多 GPU 支持
### DistributedDataParallel (DDP) — 推荐

In [ ]:
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

# 初始化进程组
dist.init_process_group(backend='nccl')

# 包装模型
model = DDP(model, device_ids=[local_rank])

# 每个进程处理自己的数据分片
output = model(input)
loss = criterion(output, target)
loss.backward()

### FullyShardedDataParallel (FSDP) — 大模型

In [ ]:
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

model = FSDP(model, auto_wrap_policy=...)

### 多 GPU 显存管理

In [ ]:
# 每个 GPU 独立的显存统计
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.memory_allocated(i) / 1e9:.2f} GB")

## 自动混合精度 (AMP)
torch.cuda.amp 提供自动混合精度训练，在保持模型精度的同时加速训练并减少显存占用。

核心原理：
- 前向传播：自动将部分运算转为 FP16/BF16
- 反向传播：缩放损失值避免梯度下溢
- 权重更新：在 FP32 主权重上执行，保证精度


In [ ]:
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()

for data, target in dataloader:
    optimizer.zero_grad()
    
    with autocast():  # 自动选择 FP16/BF16 进行前向传播
        output = model(data)
        loss = criterion(output, target)
    
    # 缩放梯度防止 FP16 下溢
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

## NVTX 性能分析
- 配合 Nsight Systems (nsys profile) 使用，可以在时间线上清晰看到标记的代码段。

In [ ]:
from torch.cuda import nvtx

# 手动标记代码范围
nvtx.range_push("Forward Pass")
output = model(input)
nvtx.range_pop()

# 或使用上下文管理器
with nvtx.range("Backward Pass"):
    loss.backward()

## 关键概念速查表

| API                             | 作用                |
| ------------------------------- | ----------------- |
| `torch.cuda.is_available()`     | 检测 CUDA 是否可用      |
| `torch.cuda.device_count()`     | GPU 数量            |
| `torch.cuda.set_device(i)`      | 设置当前 GPU          |
| `torch.cuda.synchronize()`      | 阻塞等待所有 GPU 操作完成   |
| `torch.cuda.Stream()`           | 创建 CUDA 流         |
| `torch.cuda.Event()`            | 创建 CUDA 事件（计时/同步） |
| `torch.cuda.memory_allocated()` | 已分配显存             |
| `torch.cuda.memory_reserved()`  | 缓存池保留显存           |
| `torch.cuda.empty_cache()`      | 清空显存缓存池           |
| `torch.cuda.amp.autocast()`     | 自动混合精度上下文         |
| `torch.cuda.nvtx.range_push()`  | NVTX 标记开始         |


## 常见问题
- Q: nvidia-smi 显示的显存占用比 torch.cuda.memory_allocated() 大很多？
  - 因为 PyTorch 的缓存池保留了已释放但未归还的显存块。使用 torch.cuda.memory_reserved() 查看更接近 nvidia-smi 的值。
- Q: 为什么 torch.cuda.empty_cache() 后显存没有立刻下降？
  - <font color='red'>empty_cache() 只释放未被引用的缓存块。如果有张量仍在引用，显存不会释放。</font>
- Q: 多 GPU 训练中如何设置每个进程独占一个 GPU？
  - 使用 torch.cuda.set_device(local_rank)，并确保 CUDA_VISIBLE_DEVICES 正确设置。
- Q: CUDA OOM 如何调试？
  - 使用 TORCH_CUDA_ALLOC_CONF=expandable_segments:True 减少碎片，或启用显存分析器定位泄漏。

# 多 GPU 训练中如何设置每个进程独占一个 GPU？

## 方式 1：环境变量 CUDA_VISIBLE_DEVICES（推荐用于 torchrun/mpirun）

In [ ]:
# 启动脚本：launch.sh
#!/bin/bash

# 8 个进程，每个进程只看到 1 个 GPU
# torchrun 会自动设置 LOCAL_RANK 环境变量
torchrun \
    --nproc_per_node=8 \      # 每个节点启动 8 个进程
    --nnodes=2 \              # 共 2 个节点
    --node_rank=0 \           # 当前是节点 0
    --master_addr="192.168.1.1" \
    --master_port=29500 \
    train.py

In [ ]:
# train.py
import os
import torch
import torch.distributed as dist

def setup():
    # torchrun 自动设置的环境变量：
    # LOCAL_RANK = 0~7 (节点内进程编号)
    # RANK = 0~15 (全局进程编号)
    # WORLD_SIZE = 16 (总进程数)
    
    local_rank = int(os.environ['LOCAL_RANK'])
    rank = int(os.environ['RANK'])
    world_size = int(os.environ['WORLD_SIZE'])
    
    # 每个进程绑定到一个 GPU
    torch.cuda.set_device(local_rank)  # ← 关键！进程 i 绑定 GPU i
    
    # 初始化进程组
    dist.init_process_group(
        backend='nccl',
        rank=rank,
        world_size=world_size
    )
    
    print(f"[Rank {rank}] Local rank {local_rank} → GPU {local_rank}")
    print(f"[Rank {rank}] GPU name: {torch.cuda.get_device_name(local_rank)}")

def train():
    setup()
    
    # 创建模型并放到当前进程的 GPU
    model = MyModel().cuda()  # 自动使用 set_device 设置的 GPU
    
    # 包装为 DDP
    model = torch.nn.parallel.DistributedDataParallel(
        model, 
        device_ids=[int(os.environ['LOCAL_RANK'])]  # 指定当前 GPU
    )
    
    # 训练...
    for batch in dataloader:
        loss = model(batch.cuda())
        loss.backward()
        optimizer.step()

if __name__ == '__main__':
    train()

In [ ]:
输出：

[Rank 0] Local rank 0 → GPU 0
[Rank 0] GPU name: NVIDIA A100-SXM4-80GB
[Rank 1] Local rank 1 → GPU 1
[Rank 1] GPU name: NVIDIA A100-SXM4-80GB
...
[Rank 7] Local rank 7 → GPU 7

## 方式 3：Megatron-LM 中的设置（实际框架用法）

In [ ]:
# Megatron-LM 的初始化逻辑（简化版）
import os
import torch
from megatron.core import parallel_state

def initialize_megatron():
    # 从环境变量获取并行信息
    local_rank = int(os.environ.get('LOCAL_RANK', 0))
    rank = int(os.environ.get('RANK', 0))
    world_size = int(os.environ.get('WORLD_SIZE', 1))
    
    # 1. 绑定 GPU（核心步骤）
    torch.cuda.set_device(local_rank)
    
    # 2. 初始化 torch.distributed
    torch.distributed.init_process_group(
        backend='nccl',
        rank=rank,
        world_size=world_size
    )
    
    # 3. 初始化 Megatron 并行状态
    parallel_state.initialize_model_parallel(
        tensor_model_parallel_size=2,    # TP=2
        pipeline_model_parallel_size=4,  # PP=4
        # DP = world_size / (TP × PP) = 8 / 8 = 1
    )
    
    # 验证绑定
    print(f"Rank {rank}: using GPU {local_rank}")
    print(f"TP group: {parallel_state.get_tensor_model_parallel_group()}")
    print(f"PP group: {parallel_state.get_pipeline_model_parallel_group()}")

# 启动命令（同 torchrun）
# torchrun --nproc_per_node=8 train_megatron.py